# UC01 — Puntuación de Fraude en Siniestros

## Objetivos de Aprendizaje

En este notebook aprenderás a:

1. **Crear un flujo ML completo** usando solo SQL en Snowflake
2. **Usar `SNOWFLAKE.ML.CLASSIFICATION`** para construir un modelo de clasificación binaria
3. **Generar datos sintéticos** que simulan siniestros reales de seguros
4. **Construir variables predictoras** a partir de múltiples fuentes de datos
5. **Evaluar el rendimiento del modelo** con métricas integradas
6. **Puntuar nuevos siniestros** y generar predicciones en tiempo real
7. **Construir dashboards interactivos** con Streamlit

---

## Qué Construirás

Un **modelo de clasificación binaria** que predice si un siniestro es fraudulento (1) o legítimo (0) basándose en:
- Perfil del tomador (historial de siniestros, antigüedad, tipo de póliza)
- Características del siniestro (importe, tiempo de notificación, testigos)
- Señales de red (taller, perito, proveedor médico)
- Patrones de comportamiento (hora del accidente, día de la semana)

**Valor de negocio**: Identificar siniestros fraudulentos antes del pago, reduciendo pérdidas hasta un 40%.

---

## Requisitos Técnicos

- **Cuenta Snowflake** con acceso SQL estándar
- **Cortex ML activado**
- **Aproximadamente 12 minutos** para completar
- **Sin conocimientos de Python** — 100% SQL (excepto el dashboard Streamlit)

---

## Funcionalidades Snowflake Utilizadas

- `GENERATOR()` — Crear datos sintéticos realistas
- `CREATE SNOWFLAKE.ML.CLASSIFICATION` — Entrenar modelo de clasificación
- `!PREDICT()` — Generar predicciones sobre nuevos siniestros
- `!SHOW_EVALUATION_METRICS()` — Ver rendimiento del modelo
- Funciones de ventana — Calcular historial y patrones
- Streamlit — Dashboard interactivo

---

## Paso 1: Configuración del Entorno

Creamos objetos Snowflake aislados para este tutorial:
- **Base de datos**: `FRAUDE_SINIESTROS_DB`
- **Esquema**: `PUBLIC`
- **Warehouse**: `COMPUTE_WH`

In [ ]:
CREATE DATABASE IF NOT EXISTS FRAUDE_SINIESTROS_DB;
CREATE SCHEMA IF NOT EXISTS FRAUDE_SINIESTROS_DB.PUBLIC;
USE SCHEMA FRAUDE_SINIESTROS_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS base_datos, CURRENT_SCHEMA() AS esquema, CURRENT_WAREHOUSE() AS warehouse;

---

## Paso 2: Generar Datos Sintéticos de Siniestros

Creamos datos sintéticos realistas para **1.000 siniestros** en dos fuentes:
1. **Tomadores** — Datos demográficos, antigüedad, historial de siniestros previos
2. **Siniestros** — Fecha, importe declarado, tiempo de notificación, testigos, taller

### Patrones Simulados

- **Siniestros legítimos (70%)**: Notificación rápida, importes coherentes, talleres oficiales
- **Siniestros fraudulentos (30%)**: Notificación tardía, importes inflados, talleres sospechosos, accidentes nocturnos

In [ ]:
CREATE OR REPLACE TABLE tomadores AS
SELECT
    'TOM' || LPAD(SEQ4(), 5, '0') AS tomador_id,
    UNIFORM(20, 75, RANDOM()) AS edad,
    CASE WHEN UNIFORM(0,1,RANDOM()) < 0.5 THEN 'M' ELSE 'F' END AS genero,
    CASE
        WHEN UNIFORM(1,100,RANDOM()) <= 40 THEN 'Auto'
        WHEN UNIFORM(1,100,RANDOM()) <= 70 THEN 'Hogar'
        ELSE 'Vida'
    END AS tipo_poliza,
    UNIFORM(1, 15, RANDOM()) AS anos_como_cliente,
    UNIFORM(0, 5, RANDOM()) AS siniestros_previos,
    DATEADD(day, -UNIFORM(365, 5000, RANDOM()), CURRENT_DATE()) AS fecha_alta
FROM TABLE(GENERATOR(ROWCOUNT => 500));

SELECT * FROM tomadores LIMIT 10;

In [ ]:
CREATE OR REPLACE TABLE siniestros AS
WITH base AS (
    SELECT
        'SIN' || LPAD(SEQ4(), 6, '0') AS siniestro_id,
        t.tomador_id,
        CASE WHEN UNIFORM(0,1,RANDOM()) < 0.30 THEN 1 ELSE 0 END AS es_fraude,
        DATEADD(day, -UNIFORM(1, 730, RANDOM()), CURRENT_DATE()) AS fecha_accidente
    FROM tomadores t CROSS JOIN TABLE(GENERATOR(ROWCOUNT => 2))
)
SELECT
    siniestro_id, tomador_id, es_fraude, fecha_accidente,
    CASE WHEN es_fraude=1 THEN UNIFORM(8,45,RANDOM()) ELSE UNIFORM(1,7,RANDOM()) END AS dias_hasta_notificacion,
    CASE WHEN es_fraude=1 THEN UNIFORM(3000,25000,RANDOM()) ELSE UNIFORM(300,8000,RANDOM()) END::DECIMAL(10,2) AS importe_declarado,
    CASE WHEN es_fraude=1 THEN UNIFORM(0,1,RANDOM()) ELSE UNIFORM(0,3,RANDOM()) END AS num_testigos,
    CASE WHEN es_fraude=1 THEN UNIFORM(20,23,RANDOM()) ELSE UNIFORM(7,19,RANDOM()) END AS hora_accidente,
    CASE WHEN es_fraude=1 THEN 'TALLER_SOSPECHOSO_' || UNIFORM(1,5,RANDOM()) ELSE 'TALLER_OFICIAL_' || UNIFORM(1,20,RANDOM()) END AS taller_id
FROM base;

SELECT * FROM siniestros LIMIT 10;

---

## Paso 3: Ingeniería de Variables Predictoras

Transformamos datos brutos en **variables listas para ML**:

1. **Variables del siniestro**: dias_hasta_notificacion, importe_declarado, num_testigos, hora_accidente, taller_sospechoso
2. **Variables del tomador**: edad, antiguedad, siniestros_previos, ratio_siniestros_ano, tipo_poliza

In [ ]:
CREATE OR REPLACE TABLE siniestros_features AS
SELECT
    s.siniestro_id,
    s.dias_hasta_notificacion::FLOAT AS dias_hasta_notificacion,
    s.importe_declarado::FLOAT AS importe_declarado,
    s.num_testigos::FLOAT AS num_testigos,
    s.hora_accidente::FLOAT AS hora_accidente,
    CASE WHEN s.taller_id LIKE 'TALLER_SOSPECHOSO%' THEN 1.0 ELSE 0.0 END AS taller_sospechoso,
    DAYOFWEEK(s.fecha_accidente)::FLOAT AS dia_semana_accidente,
    MONTH(s.fecha_accidente)::FLOAT AS mes_accidente,
    t.edad::FLOAT AS edad_tomador,
    t.anos_como_cliente::FLOAT AS anos_como_cliente,
    t.siniestros_previos::FLOAT AS siniestros_previos,
    CASE WHEN t.tipo_poliza='Auto' THEN 1.0 ELSE 0.0 END AS poliza_auto,
    CASE WHEN t.tipo_poliza='Hogar' THEN 1.0 ELSE 0.0 END AS poliza_hogar,
    (t.siniestros_previos / NULLIF(t.anos_como_cliente,0))::FLOAT AS ratio_siniestros_ano,
    s.es_fraude
FROM siniestros s JOIN tomadores t ON s.tomador_id = t.tomador_id;

SELECT * FROM siniestros_features LIMIT 10;

---

## Paso 4: Preparar Datos de Entrenamiento y Test

Dividimos en entrenamiento (80%) y test (20%).

In [ ]:
CREATE OR REPLACE TABLE entrenamiento AS SELECT * FROM siniestros_features SAMPLE (80);
CREATE OR REPLACE TABLE test AS SELECT * FROM siniestros_features WHERE siniestro_id NOT IN (SELECT siniestro_id FROM entrenamiento);

SELECT 'Entrenamiento' AS conjunto, COUNT(*) AS total, SUM(es_fraude) AS fraudes, ROUND(SUM(es_fraude)/COUNT(*)*100,1)||'%' AS tasa_fraude FROM entrenamiento
UNION ALL
SELECT 'Test', COUNT(*), SUM(es_fraude), ROUND(SUM(es_fraude)/COUNT(*)*100,1)||'%' FROM test;

---

## Paso 5: Entrenar el Modelo de Detección de Fraude

`SNOWFLAKE.ML.CLASSIFICATION` es la función AutoML nativa de Snowflake:
- Prueba automáticamente múltiples algoritmos (XGBoost, Random Forest, etc.)
- Sin Python — 100% SQL
- Evaluación integrada

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION detector_fraude(
    INPUT_DATA     => SYSTEM$REFERENCE('TABLE', 'entrenamiento'),
    TARGET_COLNAME => 'es_fraude',
    CONFIG_OBJECT  => {'evaluate': TRUE}
);

---

## Paso 6: Evaluar el Rendimiento del Modelo

- **Exactitud (Accuracy)**: % predicciones correctas (objetivo: >75%)
- **Precisión**: De los marcados como fraude, % que realmente lo son (objetivo: >70%)
- **Sensibilidad (Recall)**: De los fraudes reales, % detectados (objetivo: >80%) — la más importante
- **F1 Score**: Media armónica (objetivo: >0.75)

In [ ]:
CALL detector_fraude!SHOW_EVALUATION_METRICS();

In [ ]:
CALL detector_fraude!SHOW_CONFUSION_MATRIX();

In [ ]:
CALL detector_fraude!SHOW_FEATURE_IMPORTANCE();

---

## Paso 7: Puntuar Nuevos Siniestros

Usamos el modelo para predecir fraude en el conjunto de test.

### Estratificación de Riesgo
- **Alto Riesgo** (prob. fraude >= 70%): Derivar a investigación
- **Riesgo Medio** (30-70%): Revisión por perito senior
- **Bajo Riesgo** (<30%): Tramitación normal

In [ ]:
CREATE OR REPLACE TABLE predicciones_fraude AS
SELECT
    siniestro_id,
    detector_fraude!PREDICT(OBJECT_CONSTRUCT(
        'dias_hasta_notificacion', dias_hasta_notificacion,
        'importe_declarado', importe_declarado,
        'num_testigos', num_testigos,
        'hora_accidente', hora_accidente,
        'taller_sospechoso', taller_sospechoso,
        'dia_semana_accidente', dia_semana_accidente,
        'mes_accidente', mes_accidente,
        'edad_tomador', edad_tomador,
        'anos_como_cliente', anos_como_cliente,
        'siniestros_previos', siniestros_previos,
        'poliza_auto', poliza_auto,
        'poliza_hogar', poliza_hogar,
        'ratio_siniestros_ano', ratio_siniestros_ano
    )) AS prediccion,
    prediccion['class']::INT AS fraude_predicho,
    prediccion['probability']['1']::FLOAT AS probabilidad_fraude,
    CASE
        WHEN prediccion['probability']['1']::FLOAT >= 0.70 THEN 'Alto Riesgo'
        WHEN prediccion['probability']['1']::FLOAT >= 0.30 THEN 'Riesgo Medio'
        ELSE 'Bajo Riesgo'
    END AS nivel_riesgo,
    es_fraude AS fraude_real
FROM test;

SELECT siniestro_id, fraude_predicho, ROUND(probabilidad_fraude,3) AS prob_fraude, nivel_riesgo, fraude_real
FROM predicciones_fraude ORDER BY probabilidad_fraude DESC LIMIT 20;

In [ ]:
SELECT nivel_riesgo, COUNT(*) AS total, SUM(fraude_real) AS fraudes_reales,
    ROUND(SUM(fraude_real)/COUNT(*)*100,1)||'%' AS tasa_fraude_real,
    ROUND(SUM(CASE WHEN fraude_predicho=fraude_real THEN 1 ELSE 0 END)/COUNT(*)*100,1)||'%' AS exactitud
FROM predicciones_fraude GROUP BY nivel_riesgo
ORDER BY CASE nivel_riesgo WHEN 'Alto Riesgo' THEN 1 WHEN 'Riesgo Medio' THEN 2 ELSE 3 END;

---

## Paso 8: Dashboard Interactivo con Streamlit

Dashboard mostrando KPIs, distribución de riesgo y lista de siniestros con filtros.

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title("Detección de Fraude en Siniestros")
st.markdown("### Puntuación ML en Tiempo Real — Snowflake Cortex")

df = session.table("predicciones_fraude").to_pandas()

col1, col2, col3, col4 = st.columns(4)
with col1: st.metric("Total Siniestros", len(df))
with col2: st.metric("Alto Riesgo", len(df[df["NIVEL_RIESGO"]=="Alto Riesgo"]))
with col3: st.metric("Exactitud", f"{(df['FRAUDE_PREDICHO']==df['FRAUDE_REAL']).mean():.1%}")
with col4: st.metric("Tasa Fraude Real", f"{df['FRAUDE_REAL'].mean():.1%}")

st.markdown("---")
st.subheader("Distribución por Nivel de Riesgo")
rc = df["NIVEL_RIESGO"].value_counts()
st.bar_chart(pd.DataFrame({"Siniestros": rc.values}, index=rc.index))

st.markdown("---")
st.subheader("Siniestros por Nivel de Riesgo")
filtro = st.multiselect("Filtrar", ["Alto Riesgo","Riesgo Medio","Bajo Riesgo"], default=["Alto Riesgo","Riesgo Medio","Bajo Riesgo"])
fdf = df[df["NIVEL_RIESGO"].isin(filtro)].copy()
fdf["Prob. Fraude %"] = fdf["PROBABILIDAD_FRAUDE"].apply(lambda x: f"{x:.1%}")
st.dataframe(fdf[["SINIESTRO_ID","Prob. Fraude %","NIVEL_RIESGO","FRAUDE_PREDICHO","FRAUDE_REAL"]].sort_values("PROBABILIDAD_FRAUDE",ascending=False), use_container_width=True, height=400)

st.markdown("---")
csv = df[df["NIVEL_RIESGO"]=="Alto Riesgo"].to_csv(index=False)
st.download_button("Descargar Alto Riesgo (CSV)", data=csv, file_name="siniestros_alto_riesgo.csv", mime="text/csv")
st.caption("Desarrollado con Snowflake Cortex ML + Streamlit | Datos: Sintéticos")

---

## Paso 9: Limpieza del Entorno (Opcional)

Descomenta las líneas para eliminar todos los objetos creados.

In [ ]:
-- DROP DATABASE IF EXISTS FRAUDE_SINIESTROS_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;